In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import random
import itertools
import numpy as np
import pandas as pd
import psycopg2
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Patch
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:


# ==============================================================================
# 1. PREDEFINED VARIABLES & CONFIGURATION
# ==============================================================================

# --- File Paths ---
PCA_DATA_PATH = "/output/NND_PCs_18_3_2026.tsv"
GEOMAP_ZIP_PATH = "/output/gadm41_NLD_shp.zip"
ZIP_CODES_PATH = "/output/PlaatsnamenNND.xlsx"
CONVERSION_TABLE_PATH = "/output/conversion_table_19_02_2026.xlsx"
GEO_COORDS_PATH = "/output/6pp.csv"
POPULATION_DATA_PATH = "/output/geo_data/province_population.csv"
OUTPUT_DIR = "output"

# --- Database Config ---
DB_NAME = "NND"
DB_USER = ""
DB_HOST = "127.0.0.1"
DB_PORT = 5432

# --- Analysis Parameters ---
K_RANGE = range(7, 8)  # Range of cluster sizes (K) to evaluate
RELEVANT_PCS = ["PC1", "PC2", "PC3", "PC4", "PC6", "PC7", "PC8", "PC9"]
RANDOM_SEED = 42

# --- Style Palette & Plot Configuration ---
CLUSTER_COLORS = [
    "salmon", "palegoldenrod", "lightskyblue", "palevioletred",
    "lightgreen", "burlywood", "lavender", "thistle"
]
PIE_OFFSETS = {
    "Zeeland": (0, 0.2),
    "Noord-Brabant": (0, 0.1),
    "Groningen": (-0.27, 0.15)
}

# Ensure text formats remain vector-editable when saving to PDF
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42


In [ ]:

# ==============================================================================
# 2. REQUIRED DATA (INPUT)
# ==============================================================================
"""
REQUIRED INPUT DATA:
1. PCA components tsv file (`NND_PCs_18_3_2026.tsv`)
2. Shapefile zip for the geographic boundaries (`gadm41_NLD_shp.zip`)
3. Postal conversion table (`PlaatsnamenNND.xlsx` and `conversion_table_19_02_2026.xlsx`)
4. Lat/Lon mapping data (`6pp.csv`)
5. Population statistics per province (`province_population.csv`)
"""

# --- Load Geographic Boundaries ---
print("Loading Shapefile map data...")
mapdf = gpd.read_file(GEOMAP_ZIP_PATH, layer="gadm41_NLD_1")
mapdf["NAME_1"] = mapdf["NAME_1"].replace({"Fryslân": "Friesland", "NA": "Zuid-Holland"})

# --- Load PCA Components ---
print("Loading PCA components...")
PCA_clean_df = pd.read_csv(PCA_DATA_PATH, sep="\t")
pc_cols = [col for col in PCA_clean_df.columns if col.startswith("PC")]
scaler = StandardScaler()
PCA_clean_df[pc_cols] = scaler.fit_transform(PCA_clean_df[pc_cols])

# --- Load Database Geosocial Factors ---
try:
    print("Connecting to local database for geosocial metrics...")
    conn = psycopg2.connect(database=DB_NAME, user=DB_USER, host=DB_HOST, port=DB_PORT, password='')
    cur = conn.cursor()
    cur.execute("select * from geosocial_factors")
    records = cur.fetchall()
    geosocial_factors = pd.DataFrame(records, columns=[desc[0] for desc in cur.description])
    cur.close()
    conn.close()
except Exception as e:
    print(f"Warning: Could not fetch from database. Using fallback local path if exists. Error: {e}")
    # Placeholder to read local data if database fails
    # geosocial_factors = pd.read_csv("output/geosocial_factors.csv")

geosocial_factors["province"] = geosocial_factors["province"].replace({
    "NH": "Noord-Holland", 
    "ZH": "Zuid-Holland", 
    "Brabant": "Noord-Brabant"
})

# --- Load Local Zip-Codes & Coordinate tables ---
print("Loading and mapping zip code conversion tables...")
zip_codes_nnd = pd.read_excel(ZIP_CODES_PATH, header=None, names=["City", "ZipCode", "donor_id"])
zip_codes_nnd["donor_id"] = "NBB " + zip_codes_nnd["donor_id"]

conversion_df = pd.read_excel(CONVERSION_TABLE_PATH)
conversion_dict = pd.Series(conversion_df.Identifier.values, index=conversion_df.NBB_nr).to_dict()
zip_codes_nnd["donor_id"] = zip_codes_nnd["donor_id"].map(conversion_dict)

geo_info_nl = pd.read_csv(GEO_COORDS_PATH)
subset_geo_df = geo_info_nl[["plaats", "postcode", "latitude", "longitude"]].copy()
subset_geo_df["plaats"] = subset_geo_df["plaats"].str.lower()
subset_geo_df = subset_geo_df.drop_duplicates(subset='plaats', keep='first')

zip_codes_nnd["City"] = zip_codes_nnd["City"].str.lower()


# ==============================================================================
# 3. HELPER FUNCTIONS
# ==============================================================================

def truncate_colormap(cmap, minval=0.0, maxval=0.5, n=100):
    """Truncates a colormap range to keep colors light/distinguishable."""
    return mcolors.LinearSegmentedColormap.from_list(
        f"trunc({cmap.name},{minval},{maxval})",
        cmap(np.linspace(minval, maxval, n))
    )


def get_inset_position(ax, row, province_name):
    """Calculates coordinate transformation for placing inset charts on map polygons."""
    x, y = row["geometry"].representative_point().coords[0]

    if province_name in PIE_OFFSETS:
        dx, dy = PIE_OFFSETS[province_name]
        x += dx
        y += dy

    x_disp, y_disp = ax.transData.transform((x, y))
    return ax.transAxes.inverted().transform((x_disp, y_disp))


def run_clustering_pipeline(
    k,
    X,
    donors_gdf,
    mapdf,
    normalized_per_province,
    output_base=OUTPUT_DIR
):
    """Runs K-means, performs normalisation merging, and produces geographic plots."""
    print(f"Running clustering pipeline for k = {k}")
    
    # ---------------------------
    # Establish Output Folder
    # ---------------------------
    k_folder = os.path.join(output_base, f"k_{k:02d}")
    os.makedirs(k_folder, exist_ok=True)

    # ---------------------------
    # KMeans Clustering Execution
    # ---------------------------
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED)
    clusters = kmeans.fit_predict(X)

    donors_gdf = donors_gdf.copy()
    donors_gdf["cluster"] = clusters

    # ---------------------------
    # Prepare Map Geometry Layer
    # ---------------------------
    mapdf_copy = mapdf.copy()
    norm_prov = normalized_per_province.copy()
    norm_prov.columns = ["NAME_1", "normalized_score"]

    mapdf_copy = mapdf_copy.merge(norm_prov, on="NAME_1", how="left")

    # ---------------------------
    # Colormap Assignments
    # ---------------------------
    cmap = truncate_colormap(cm.Blues, 0.0, 0.5)
    norm = mcolors.Normalize(
        vmin=mapdf_copy["normalized_score"].min(),
        vmax=mapdf_copy["normalized_score"].max()
    )

    mapdf_copy["color"] = mapdf_copy["normalized_score"].apply(
        lambda val: cmap(norm(val)) if not np.isnan(val) else "#f5f5f5"
    )
    mapdf_copy.loc[mapdf_copy["NAME_1"].isin(["IJsselmeer", "Zeeuwse meren"]), "color"] = "white"

    # Legend setups
    cluster_labels = [f"Cluster {i}" for i in range(k)]
    legend_handles = [
        Patch(facecolor=CLUSTER_COLORS[i], edgecolor="black", label=cluster_labels[i])
        for i in range(k)
    ]

    # ==========================================================================
    # VISUALIZATION 1: Map with Jittered Donor Locations
    # ==========================================================================
    fig, ax = plt.subplots(figsize=(8, 10))
    mapdf_copy.plot(ax=ax, color=mapdf_copy["color"], edgecolor="black")

    donors_gdf.plot(
        ax=ax,
        column="cluster",
        markersize=20,
        categorical=True,
        color=[CLUSTER_COLORS[c] for c in donors_gdf["cluster"]]
    )

    # Attach Normalized Heat Scale Colorbar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Normalized score")

    plt.title(f"K-means clusters (k = {k})")
    ax.legend(
        handles=legend_handles,
        title="Clusters",
        loc="upper left",
        bbox_to_anchor=(0.1, 0.75)
    )
    ax.set_axis_off()
    plt.savefig(os.path.join(k_folder, "map_clusters.png"), bbox_inches="tight")
    plt.close()

    # ==========================================================================
    # VISUALIZATION 2: Map with Province Pie Charts
    # ==========================================================================
    fig, ax = plt.subplots(figsize=(10, 12))
    mapdf_copy.plot(ax=ax, color=mapdf_copy["color"], edgecolor="black")

    cluster_counts = donors_gdf.groupby(["province", "cluster"]).size().unstack(fill_value=0)

    # Attach Scale Colorbar
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Normalized score")

    centroids = {}

    for _, row in mapdf_copy.iterrows():
        province_name = row["NAME_1"]
        if province_name not in cluster_counts.index:
            continue

        counts = cluster_counts.loc[province_name]
        centroids[province_name] = row.geometry.centroid
        x_axes, y_axes = get_inset_position(ax, row, province_name)

        # Build Inset sub-plot coordinates
        ax_inset = inset_axes(
            ax,
            width=0.6,
            height=0.6,
            loc="center",
            bbox_to_anchor=(x_axes, y_axes),
            bbox_transform=ax.transAxes,
            borderpad=0,
        )

        ax_inset.pie(
            counts,
            colors=CLUSTER_COLORS[:len(counts)],
            wedgeprops=dict(edgecolor="black", linewidth=0.8)
        )
        ax_inset.axis("off")

    # --- Manual Annotation Callouts ---
    cluster6_color = "thistle"
    cluster4_color = "lightgreen"

    if "Noord-Holland" in centroids:
        c = centroids["Noord-Holland"]
        ax.annotate(
            "    Cluster 6 (n=2)",
            xy=(c.x, c.y),
            xytext=(c.x - 1.25, c.y + 0.01),
            arrowprops=dict(arrowstyle="-", lw=1.5, shrinkB=0, relpos=(0, 1.35)),
            fontsize=12,
            ha="left",
            bbox=dict(boxstyle="round,pad=0.8", fc="white", ec="black")
        )
        ax.scatter(
            c.x - 1.22, c.y + 0.029,
            s=180, marker="s", color=cluster6_color, edgecolor="black", zorder=5
        )

    if "Utrecht" in centroids:
        c = centroids["Utrecht"]
        ax.annotate(
            "    Cluster 4 (n=1)",
            xy=(c.x, c.y),
            xytext=(c.x - 1.61, c.y + 0.23),
            arrowprops=dict(arrowstyle="-", lw=1.5, shrinkB=0),
            fontsize=12,
            ha="left",
            bbox=dict(boxstyle="round,pad=0.8", fc="white", ec="black")
        )
        ax.scatter(
            c.x - 1.58, c.y + 0.25,
            s=180, marker="s", color=cluster4_color, edgecolor="black", zorder=5
        )

    ax.legend(
        handles=legend_handles,
        title="Clusters",
        loc="upper left",
        bbox_to_anchor=(-0.02, 0.85),
        ncol=2
    )
    ax.set_axis_off()
    plt.savefig(os.path.join(k_folder, "map_clusters_pie.pdf"), bbox_inches="tight")
    plt.close()


# ==============================================================================
# 4. PROCESSES
# ==============================================================================

# --- Process 1: Database and PCA Merging ---
pca_plus_geo_df = geosocial_factors.merge(PCA_clean_df, on="donor_id")
pca_plus_geo_df = pca_plus_geo_df[["donor_id", "province", "region"] + RELEVANT_PCS]

merged_df1 = zip_codes_nnd.merge(subset_geo_df, left_on="City", right_on="plaats", how="left")
map_df_NND = merged_df1.merge(pca_plus_geo_df, on="donor_id")

# --- Process 2: GeoDataFrame Instantiation & Jitter Normalization ---
print("Creating geospatial coordinates with Jitter normalization...")
geometry = gpd.points_from_xy(map_df_NND["longitude"], map_df_NND["latitude"])
donors_gdf = gpd.GeoDataFrame(
    map_df_NND,
    geometry=geometry,
    crs="EPSG:4326"
)

# Apply coordinate jitter to avoid overlapping point issues on the visual map
np.random.seed(RANDOM_SEED)
donors_gdf["lon_jitter"] = donors_gdf["longitude"] + np.random.normal(0, 0.01, len(donors_gdf))
donors_gdf["lat_jitter"] = donors_gdf["latitude"] + np.random.normal(0, 0.01, len(donors_gdf))
donors_gdf["geometry"] = gpd.points_from_xy(donors_gdf["lon_jitter"], donors_gdf["lat_jitter"])

# Match coordinate reference systems
mapdf = mapdf.to_crs(donors_gdf.crs)

# Isolate target metrics for clustering steps
X = pca_plus_geo_df[RELEVANT_PCS].copy()

# --- Process 3: Demographics and Cohort Normalisation ---
print("Calculating normalized statistics across Dutch population provinces...")
province_population = pd.read_csv(POPULATION_DATA_PATH)

donor_counts = donors_gdf["province"].value_counts().reset_index()
donor_counts.columns = ["Province", "donor_count"]

# Combine donor tracking metrics with population databases
normalized_merged_df = donor_counts.merge(province_population, on="Province", how="left")
normalized_merged_df["donors_per_100k"] = (
    normalized_merged_df["donor_count"] / normalized_merged_df["population"]
) * 100000

normalized_per_province = normalized_merged_df[["Province", "donors_per_100k"]]


# ==============================================================================
# 5. OUTPUT: Run Visualization Pipelines
# ==============================================================================

print("Starting pipeline plotting execution...")
for k in K_RANGE:
    run_clustering_pipeline(
        k=k,
        X=X,
        donors_gdf=donors_gdf,
        mapdf=mapdf,
        normalized_per_province=normalized_per_province
    )
print("Processing Completed successfully.")